<div style="font-variant: small-caps; 
      font-weight: normal; 
      font-size: 37px; 
      text-align: center; 
      padding: 15px; 
      margin: 10px;">
  Machine Learning<br>
  </div> 

<div style="font-variant: small-caps; 
      font-weight: normal; 
      font-size: 35px; 
      text-align: center; 
      padding: 15px; 
      margin: 10px;">
  -<br>
  </div> 
  
<div style="font-variant: small-caps; 
      font-weight: normal; 
      font-size: 35px; 
      text-align: center; 
      padding: 15px; 
      margin: 10px;">
      Illustration with Random Forests<br><br>
  </div> 

<div style="font-variant: small-caps; 
      font-weight: normal; 
      font-size: 25px; 
      text-align: center; 
      padding: 15px; 
      margin: 10px;">
      <font color=orange> 5 - Model Interpretation</font>
  </div>

<a id="plan"></a>


### Model Interpretation

-  [Random Forest features importance](#RF_feature_importance)<br>
-  [Permutation importance](#permutation_feature_importance)<br>
-  [Feature correlations](#redundant_features)<br>


[Bottom](#bottom)

### Imports

In [ ]:
# for package auto reload
%load_ext autoreload
%autoreload 2

# for better rendering of plots in jupyter notebook
%matplotlib inline

In [ ]:
# base modules
import math
from collections import defaultdict
from pathlib import Path

# for manipulating data
import numpy as np
import pandas as pd
import scipy

# for visualization
from matplotlib import pyplot as plt
from scipy.cluster import hierarchy

# for Machine Learning
from scipy.spatial.distance import squareform
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import cross_val_score

In [ ]:
from eclyon.explain import rf_feat_importance, tree_feature_importance
from eclyon.transforms import process_df, split_vals

#  Model Interpretation

[Back to top](#plan)

In [ ]:
path_to_repo = Path("..").resolve()
path_to_data = path_to_repo / "data"

In [ ]:
df_transformed = pd.read_feather(path_to_data / "transformed")

df, y, nas = process_df(df_transformed, "SalePrice")

In [ ]:
n_total = len(df)
n_valid = 12000  # same as Kaggle's test set size
n_train = n_total - n_valid
n_small = 20000

X_train, X_valid = split_vals(df, n_train)
y_train, y_valid = split_vals(y, n_train)

X_small, _ = split_vals(df, n_small)
y_small, _ = split_vals(y, n_small)

print("Number of small training data points: X = {}, y = {}".format(X_small.shape, y_small.shape))
print("Number of full training data points: X = {}, y = {}".format(X_train.shape, y_train.shape))
print("Number of validation data points: X = {}, y = {}".format(X_valid.shape, y_valid.shape))

In [ ]:
def rmse(y_gold, y_pred):
    return math.sqrt(((y_gold - y_pred) ** 2).mean())


def print_score(m, X_train, y_train, X_valid, y_valid):
    print("RMSE on train set: {:.4f}".format(rmse(m.predict(X_train), y_train)))
    print("RMSE on valid set: {:.4f}".format(rmse(m.predict(X_valid), y_valid)))
    print("R^2 on train set: {:.4f}".format(m.score(X_train, y_train)))
    print("R^2 on valid set: {:.4f}".format(m.score(X_valid, y_valid)))
    if hasattr(m, "oob_score_"):
        print("R^2 on oob set: {:.4f}".format(m.oob_score_))
    return

### $\bullet$ Random Forest feature importance

[Back to top](#plan)

It's not normally enough to just to know that a model can make accurate predictions - we also want to know *how* it's making predictions. The most important way to see this is with *feature importance*.

Scikit-learn's implementation of Random Forests carry _impurity-based_ feature importance scores (the higher, the more important the feature) in the attribute `model.feature_importances_` of a fitted model. The importance of a feature is computed as the (normalized) total reduction of the criterion brought by that feature on the training set. 

Since they are computed on the training set these scores should be taken with caution, as the importances can be high even for features that are not predictive of the target variable, as long as the model has the capacity to use them to overfit. As a result, features that are deemed of low importance for a bad model (high training score but low validation/cross-validation score) could be very important for a good model. Therefore it is always important to evaluate the predictive power of a model using a validation set (or better with cross-validation) prior to computing importances. These feature importance does not reflect to the intrinsic predictive value of a feature by itself, but how important this feature is for a particular random forest.


See more on the [scikit-learn documentation](https://scikit-learn.org/stable/modules/partial_dependence.html).

In [ ]:
model = RandomForestRegressor(
    n_estimators=50,
    min_samples_split=10,
    max_features=0.5,  # default = None
    n_jobs=-1,
    random_state=42,
    bootstrap=True,
    oob_score=True,  # default = False
    max_samples=0.75,  # default = None
)

%time model.fit(X_small, y_small)
print_score(model, X_small, y_small, X_valid, y_valid)

In [ ]:
model.feature_importances_

#### Feature importance of a single tree

In [ ]:
# pick the first tree of the forest
tree = model.estimators_[0]

We propose 3 ways to access the feature importances of a single tree, with one given by the function `tree_feature_importance` which can serve as an explanation of how these scores are computed:

In [ ]:
# method 1
tree.tree_.compute_feature_importances(normalize=True)

In [ ]:
# method 2
tree_feature_importance(tree, normalize=True)

In [ ]:
# method 3
tree.feature_importances_

In [ ]:
feat_importance = rf_feat_importance(tree, X_small)
feat_importance[:20]

In [ ]:
barh = feat_importance[:30].plot("cols", "imp", "barh", figsize=(12, 10), legend=False)
barh = barh.invert_yaxis()
barh

#### Feature importance of the whole random forest

Each tree of the random forest carries its own feature importance along the explanatory variables (variables unseen during training simply have score 0.0). The overall feature importance of the random forest is then naturally defined as the _average, feature by feature, of the importance scores computed by each tree_.

In [ ]:
feat_importance = rf_feat_importance(model, X_small)
feat_importance[:20]

In [ ]:
barh = feat_importance[:30].plot("cols", "imp", "barh", figsize=(12, 10), legend=False)
barh = barh.invert_yaxis()
barh

#### Ablation of least important features

Least important features can be discarded before re-training the random forest, where it yields a slight improvement of the model's validation $R^2$ score :

In [ ]:
to_keep = feat_importance[feat_importance.imp > 0.005].cols
len(to_keep)

In [ ]:
to_keep

In [ ]:
# perform ablation of columns with low importance
df_keep = df[to_keep].copy()

n_total = len(df_keep)
n_valid = 12000  # same as Kaggle's test set size
n_train = n_total - n_valid
n_small = 20000

X_train_keep, X_valid_keep = split_vals(df_keep, n_train)
y_train, y_valid = split_vals(y, n_train)

X_small_keep, _ = split_vals(df_keep, n_small)
y_small, _ = split_vals(y, n_small)

print("Number of small training data points: X = {}, y = {}".format(X_small.shape, y_small.shape))
print("Number of full training data points: X = {}, y = {}".format(X_train.shape, y_train.shape))
print("Number of validation data points: X = {}, y = {}".format(X_valid.shape, y_valid.shape))

In [ ]:
model = RandomForestRegressor(
    n_estimators=50,
    min_samples_split=10,
    max_features=0.5,  # default = None
    n_jobs=-1,
    random_state=42,
    bootstrap=True,
    oob_score=True,  # default = False
    max_samples=0.75,  # default = None
)

%time model.fit(X_small_keep, y_small)
print_score(model, X_small_keep, y_small, X_valid_keep, y_valid)

In [ ]:
feat_importance = rf_feat_importance(model, X_small_keep)

barh = feat_importance.plot("cols", "imp", "barh", figsize=(12, 10), legend=False)
barh = barh.invert_yaxis()
barh

### $\bullet$ Permutation importance

[Back to top](#plan)

Permutation feature importance is a model inspection technique that can be used for any fitted estimator, and is not restricted to Random Forests. This is especially useful for non-linear or opaque estimators.


The permutation feature importance is defined to be the decrease in a model score when a single feature value is randomly shuffled. This procedure breaks the relationship between the feature and the target, thus the drop in the model score is indicative of how much the model depends on the feature. Precisely, the `permutation_importance` function takes as input a fitted model and a validation set $\mathcal{D}$ with $d$ features, and compute the reference $R^2$ score sc of the model on this dataset. It then loops over a predefined number $k$ of repetitions, where during each loop $i$ a set of $d$ copies of the data are created, with in each copy $\mathcal{D}_{i, j}$ having feature $j$ being randomly shuffled (aka the term _permutation_). The $R^2$ score of the model is then computed on each of these corrupted copies of $\mathcal{D}$, yielding a set $\text{sc}_{i, j}$ of scores. The resulting importance score of each feature $j$ is then given by 

$$ \text{sc}_j = \text{sc} - \frac{1}{k}\sum_{i}\text{sc}_{i, j}$$


The importance of a feature is then higher as the model score decreases after corrupting this feature.
 


As for Random Forests impurity-based feature importance, feature importance based on permutation is of value only for good models (not overfitted and with descent cross-validated score), and does not reflect to the intrinsic predictive value of a feature by itself, but how important this feature is for a particular model.


See more on [scikit-learn documentation](https://scikit-learn.org/stable/modules/permutation_importance.html#permutation-importance)

In [ ]:
model = RandomForestRegressor(
    n_estimators=50,
    min_samples_split=10,
    max_features=0.5,  # default = None
    n_jobs=-1,
    random_state=42,
    bootstrap=True,
    oob_score=True,  # default = False
    max_samples=0.75,  # default = None
)

%time model.fit(X_small, y_small)
print_score(model, X_small, y_small, X_valid, y_valid)

In [ ]:
result = permutation_importance(
    model,
    X_valid,
    y_valid,
    n_repeats=10,
    random_state=42,
    n_jobs=1,
)
sorted_idx = result.importances_mean.argsort()

In [ ]:
result

In [ ]:
fig, ax = plt.subplots(figsize=(15, 50))
ax.boxplot(
    result.importances[sorted_idx].T,
    vert=False,
    tick_labels=X_valid.columns[sorted_idx],
)
ax.set_title("Permutation Importances (validation set)")
fig.tight_layout()
plt.show()

#### Ablation of least important features

Least important features can be discarded before re-training the random forest, where it yields a **notable improvement** of the model's validation $R^2$ score :

In [ ]:
bool_filter = result.importances_mean > 0.005
bool_filter

In [ ]:
to_keep = df.columns[bool_filter]

df_keep = df[to_keep].copy()

n_total = len(df_keep)
n_valid = 12000  # same as Kaggle's test set size
n_train = n_total - n_valid
n_small = 20000

X_train_keep, X_valid_keep = split_vals(df_keep, n_train)
y_train, y_valid = split_vals(y, n_train)

X_small_keep, _ = split_vals(df_keep, n_small)
y_small, _ = split_vals(y, n_small)

print("Number of small training data points: X = {}, y = {}".format(X_small.shape, y_small.shape))
print("Number of full training data points: X = {}, y = {}".format(X_train.shape, y_train.shape))
print("Number of validation data points: X = {}, y = {}".format(X_valid.shape, y_valid.shape))

In [ ]:
model = RandomForestRegressor(
    n_estimators=50,
    min_samples_split=10,
    max_features=0.5,  # default = None
    n_jobs=-1,
    random_state=42,
    bootstrap=True,
    oob_score=True,  # default = False
    max_samples=0.75,  # default = None
)

%time model.fit(X_small_keep, y_small)
print_score(model, X_small_keep, y_small, X_valid_keep, y_valid)

In [ ]:
result = permutation_importance(
    model,
    X_valid_keep,
    y_valid,
    n_repeats=10,
    random_state=42,
    n_jobs=1,
)
sorted_idx = result.importances_mean.argsort()

# plot boxplot
fig, ax = plt.subplots(figsize=(15, 20))
ax.boxplot(
    result.importances[sorted_idx].T,
    vert=False,
    tick_labels=X_valid.columns[sorted_idx],
)
ax.set_title("Permutation Importances (validation set)")
fig.tight_layout()
plt.show()

#### Ablation after training on full data

Such ablation may hurt performance when the training (and validation) data is abundent :

In [ ]:
model = RandomForestRegressor(
    n_estimators=50,
    max_depth=15,
    min_samples_split=10,
    max_features=0.5,  # default = None
    n_jobs=-1,
    random_state=42,
    bootstrap=True,
    oob_score=True,  # default = False
    max_samples=0.75,  # default = None
)

%time model.fit(X_train, y_train)
print_score(model, X_train, y_train, X_valid, y_valid)

In [ ]:
result = permutation_importance(
    model,
    X_valid,
    y_valid,
    n_repeats=5,
    random_state=42,
    n_jobs=1,
)
sorted_idx = result.importances_mean.argsort()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 50))
ax.boxplot(
    result.importances[sorted_idx].T,
    vert=False,
    tick_labels=X_valid.columns[sorted_idx],
)
ax.set_title("Permutation Importances (validation set)")
fig.tight_layout()
plt.show()

In [ ]:
bool_filter = result.importances_mean > 0.005
to_keep = df.columns[bool_filter]

df_keep = df[to_keep].copy()

n_total = len(df_keep)
n_valid = 12000  # same as Kaggle's test set size
n_train = n_total - n_valid
n_small = 20000

X_train_keep, X_valid_keep = split_vals(df_keep, n_train)
y_train, y_valid = split_vals(y, n_train)

X_small_keep, _ = split_vals(df_keep, n_small)
y_small, _ = split_vals(y, n_small)

print("Number of small training data points: X = {}, y = {}".format(X_small.shape, y_small.shape))
print("Number of full training data points: X = {}, y = {}".format(X_train.shape, y_train.shape))
print("Number of validation data points: X = {}, y = {}".format(X_valid.shape, y_valid.shape))

In [ ]:
model = RandomForestRegressor(
    n_estimators=50,
    max_depth=15,
    min_samples_split=10,
    max_features=0.5,  # default = None
    n_jobs=-1,
    random_state=42,
    bootstrap=True,
    oob_score=True,  # default = False
    max_samples=0.75,  # default = None
)

%time model.fit(X_train_keep, y_train)
print_score(model, X_train_keep, y_train, X_valid_keep, y_valid)

In [ ]:
# default is cv = 5
scores = cross_val_score(model, X_small_keep, y_small, cv=5, n_jobs=-1)

print(scores)
print("r2 is {:.3f} with a standard deviation of {:.3f}".format(scores.mean(), scores.std()))

### $\bullet$ Feature correlations

[Back to top](#plan)

If one computes permutation importance of features when some collinearity occurs, corrupting one feature will have little effect on the models performance because it can get the same information from a correlated feature. Because of that, the permutation importance will show that none of the features are important, which is in contradiction with a high validation that shows that some feature must be important. One approach to handling multicollinearity is by performing hierarchical clustering on the features’ Spearman rank-order correlations, picking a threshold, and keeping a single feature from each cluster.

In [ ]:
# something is weird with this column
cols = df.columns.tolist()
cols.index("saleIs_year_start")

In [ ]:
# something is weird with this column
cols = df.columns.tolist()
cols.remove("saleIs_year_start")
cols

We compute the Spearman rank-order correlation matrix of the explainatory variables using the `scipy` package. Each coefficient is a measure of the monotonicity of the relationship between two columns : it varies between -1 and +1, with 0 implying no correlation and correlations of -1 or +1 implying an exact monotonic relationship. Positive correlations imply that as x increases, so does y, whereas negative correlations imply that as x increases, y decreases.

In [ ]:
len(df.columns)

In [ ]:
corr = np.round(spearmanr(df[cols].astype(float)).correlation, 4)
corr.size

In [ ]:
col_small = [
    "Transmission",
    "Turbocharged",
    "Blade_Extension",
    "Blade_Width",
    "Enclosure_Type",
    "Engine_Horsepower",
    "Hydraulics",
    "Pushblock",
]
corr_small = np.round(scipy.stats.spearmanr(df[col_small]).correlation, 4)
corr_small.size

fig = plt.figure(figsize=(12, 12))
ax = fig.add_subplot(111)
cax = ax.matshow(corr_small)

fig.colorbar(cax)

ax.set_xticklabels([""] + col_small)
ax.set_yticklabels([""] + col_small)

In [ ]:
# corr with size = (64, 63) -> corr_condensed with size (64 * 63)/2
corr_condensed = squareform(1 - corr)
corr_condensed.size

In [ ]:
(64 * 63) / 2

In [ ]:
corr_linkage = hierarchy.linkage(corr_condensed, method="average")

In [ ]:
corr_linkage

In [ ]:
help(hierarchy.linkage)

In [ ]:
# compute plot
fig = plt.figure(figsize=(16, 20))
dendrogram = hierarchy.dendrogram(
    corr_linkage,
    labels=cols,
    orientation="left",
    leaf_font_size=16,
)
plt.show()

In [ ]:
# compute hierarchical clustering
threshold = 0.1
cluster_ids = hierarchy.fcluster(corr_linkage, threshold, criterion="distance")

In [ ]:
cluster_ids

In [ ]:
# reshape clustering as {cluster_index : feature_index}
cluster_id_to_feature_ids = defaultdict(list)
for idx, cluster_id in enumerate(cluster_ids):
    cluster_id_to_feature_ids[cluster_id].append(idx)

In [ ]:
cluster_id_to_feature_ids

In [ ]:
# keep only first feature for each cluster
selected_features = [v[0] for v in cluster_id_to_feature_ids.values()]

# shift feature index after 62
selected_features = [(i if i < 62 else i + 1) for i in selected_features]
selected_features

In [ ]:
# retained set of features
selected_features = [df.columns[i] for i in selected_features]
selected_features

In [ ]:
len(selected_features)

#### Perform training on ablated set of features

In [ ]:
df_keep = df[selected_features].copy()

n_total = len(df_keep)
n_valid = 12000  # same as Kaggle's test set size
n_train = n_total - n_valid
n_small = 20000

X_train_keep, X_valid_keep = split_vals(df_keep, n_train)
y_train, y_valid = split_vals(y, n_train)

X_small_keep, _ = split_vals(df_keep, n_small)
y_small, _ = split_vals(y, n_small)

print("Number of small training data points: X = {}, y = {}".format(X_small.shape, y_small.shape))
print("Number of full training data points: X = {}, y = {}".format(X_train.shape, y_train.shape))
print("Number of validation data points: X = {}, y = {}".format(X_valid.shape, y_valid.shape))

In [ ]:
model = RandomForestRegressor(
    n_estimators=50,
    max_depth=15,
    min_samples_split=10,
    max_features=0.5,  # default = None
    n_jobs=-1,
    random_state=42,
    bootstrap=True,
    oob_score=True,  # default = False
    max_samples=0.75,  # default = None
)

%time model.fit(X_train_keep, y_train)
print_score(model, X_train_keep, y_train, X_valid_keep, y_valid)

The model performance does not improve. Let us now see if some features shows up as important and wasn't before due to the correlation issues :

#### Permutation of feature importance after feature selection

In [ ]:
result = permutation_importance(
    model,
    X_valid_keep,
    y_valid,
    n_repeats=5,
    random_state=42,
    n_jobs=1,
)
sorted_idx = result.importances_mean.argsort()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 50))
ax.boxplot(
    result.importances[sorted_idx].T,
    vert=False,
    tick_labels=X_valid.columns[sorted_idx],
)
ax.set_title("Permutation Importances (validation set)")
fig.tight_layout()
plt.show()

**Conclusion** : We now see that `Blade_Extension`overtakes `YearMade`as the most discriminant variable. This makes a lot of sense since according to the previous dendrogram this features is extremely correlated with the feature `Engine_Horsepower`, which indeed must heavily impact the sale price.

[Back to top](#plan)